# 01 — Create Benchmark Datasets: Parquet vs Lance

**Purpose:** Generate a synthetic multimodal dataset **once** as in-memory Ray blocks, then write those same blocks to both **Parquet** and **Lance**. This isolates the storage format as the only variable for the throughput benchmark in `02_training_benchmark.ipynb`.

Covers stages **1–4** of [`README.md`](README.md): setup, generate, write, and the add-column ETL benchmark.

| | Parquet | Lance |
|---|---|---|
| Writer | Independent file per Ray task, no coordination | Fragment per task → single driver-side commit |
| Add a column | Full dataset rewrite | `add_columns` — new column only, no rewrite |

**Compute:** Databricks Classic Compute — **8 worker nodes × 16 CPUs** each.

---

**Outputs (per size tier):**
- Parquet dataset at `/Volumes/{catalog}/{schema}/{volume}/synthetic_parquet_{size}/`
- Lance dataset at `/Volumes/{catalog}/{schema}/{volume}/synthetic_lance_{size}/`

**Next:** `02_training_benchmark.ipynb`

In [ ]:
# Install packages BEFORE setup_ray_cluster — installing after shuts the cluster down.
%pip install -qU "ray[data]==2.54.0" "lance==0.17.0" "pyarrow>=16.0" Pillow numpy pandas
dbutils.library.restartPython()

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────
dbutils.widgets.dropdown("size", "10k", ["10k", "100k", "1m", "10m"], "Dataset size")
dbutils.widgets.text("catalog", "main", "UC catalog")
dbutils.widgets.text("schema", "ml_benchmark", "UC schema")
dbutils.widgets.text("volume", "lance_benchmark", "UC volume")
dbutils.widgets.text("seed", "42", "RNG seed")
dbutils.widgets.text("embedding_dim", "512", "Embedding dim")

size          = dbutils.widgets.get("size")
catalog       = dbutils.widgets.get("catalog")
schema        = dbutils.widgets.get("schema")
volume        = dbutils.widgets.get("volume")
SEED          = int(dbutils.widgets.get("seed"))
EMBEDDING_DIM = int(dbutils.widgets.get("embedding_dim"))

SIZE_MAP = {"10k": 10_000, "100k": 100_000, "1m": 1_000_000, "10m": 10_000_000}
N_ROWS   = SIZE_MAP[size]

# Fixed category set — MUST match 02_training_benchmark.ipynb.
CATEGORIES = ["cat", "dog", "car", "tree", "house", "flower", "boat", "bird"]

base_vol     = f"/Volumes/{catalog}/{schema}/{volume}"
parquet_path = f"{base_vol}/synthetic_parquet_{size}"
lance_path   = f"{base_vol}/synthetic_lance_{size}"

print(f"Size tier   : {size} ({N_ROWS:,} rows)")
print(f"Parquet out : {parquet_path}")
print(f"Lance out   : {lance_path}")
print(f"Categories  : {CATEGORIES}")

In [ ]:
import os

# Credentials — set BEFORE setup_ray_cluster so Ray workers inherit them (Ray 2.41+).
os.environ["DATABRICKS_HOST"]  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

In [ ]:
# Ensure output + Ray tmp volumes exist.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import catalog as sdk_catalog

w = WorkspaceClient()
for vol_name in [volume, "ray_tmp"]:
    try:
        w.volumes.read(f"{catalog}.{schema}.{vol_name}")
    except Exception:
        w.volumes.create(catalog_name=catalog, schema_name=schema, name=vol_name,
                         volume_type=sdk_catalog.VolumeType.MANAGED)
        print(f"Created volume {catalog}.{schema}.{vol_name}")

ray_tmp_path = f"/Volumes/{catalog}/{schema}/ray_tmp"
print(f"Ray tmp     : {ray_tmp_path}")

In [ ]:
# Classic Compute Ray cluster — 8 worker nodes × 16 CPUs.
# num_cpus_worker_node must match the Spark worker node CPU count.
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

try:
    shutdown_ray_cluster()
except Exception:
    pass

N_WORKER_NODES = 8
CPUS_PER_NODE  = 16

setup_ray_cluster(
    min_worker_nodes=N_WORKER_NODES,
    max_worker_nodes=N_WORKER_NODES,      # fixed size (min == max)
    num_cpus_worker_node=CPUS_PER_NODE,
    collect_log_to_path=ray_tmp_path,
)
ray.init(address="auto", ignore_reinit_error=True)

# Verify the whole cluster came up before submitting work.
total_cpus = ray.cluster_resources().get("CPU", 0)
alive      = sum(1 for n in ray.nodes() if n["Alive"])
print(f"Total CPUs  : {total_cpus:.0f}")
print(f"Alive nodes : {alive}")
assert total_cpus >= N_WORKER_NODES * CPUS_PER_NODE * 0.9, "Cluster did not fully start"

## Generate synthetic data (once)

`ray.data.range(N).map_batches(generate_batch)` fans generation across the cluster. Each row is seeded by `(SEED, id)`, so generation is deterministic and independent of block partitioning.

The image is **conditioned on the category** (category sets the base hue) so the classification task in `02` is learnable. Random noise is layered on so the JPEG doesn't over-compress — landing each frame in the ~30–300KB range that makes the format comparison realistic.

In [ ]:
import numpy as np


def _make_image(rng, category_idx, n_categories):
    """Procedural RGB image conditioned on category, JPEG-encoded to ~30-300KB."""
    import io
    from PIL import Image

    side = int(rng.integers(256, 512))
    base = np.zeros((side, side, 3), dtype=np.float32)
    hue = category_idx / n_categories          # category drives the base color
    base[..., 0] = 255 * hue
    base[..., 1] = 255 * (1 - hue)
    base[..., 2] = 128
    noise = rng.integers(0, 60, size=(side, side, 3))   # keeps the JPEG incompressible
    arr = np.clip(base + noise, 0, 255).astype(np.uint8)

    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="JPEG", quality=90)
    return buf.getvalue()


def generate_batch(batch, seed, categories, embedding_dim):
    ids = batch["id"]
    n_cat = len(categories)
    images, captions, embeddings, cats, brightness, quality = [], [], [], [], [], []
    for _id in ids:
        rng = np.random.default_rng([seed, int(_id)])   # per-row, deterministic
        cat_idx = int(rng.integers(0, n_cat))
        images.append(_make_image(rng, cat_idx, n_cat))
        captions.append(f"a photo of a {categories[cat_idx]} " + "x" * int(rng.integers(0, 40)))
        embeddings.append(rng.standard_normal(embedding_dim).astype(np.float32))
        cats.append(categories[cat_idx])
        brightness.append(float(rng.random()))
        quality.append(int(rng.integers(1, 6)))
    return {
        "id":         np.asarray(ids),
        "image":      np.asarray(images, dtype=object),      # inline JPEG bytes
        "caption":    np.asarray(captions, dtype=object),
        "embedding":  np.asarray(embeddings, dtype=np.float32),  # (B, dim) tensor column
        "category":   np.asarray(cats, dtype=object),
        "brightness": np.asarray(brightness, dtype=np.float32),
        "quality":    np.asarray(quality, dtype=np.int32),
    }

In [ ]:
# Generate ONCE and materialize — both formats write from these same in-memory blocks,
# so no generation noise contaminates the write-timing comparison.
override_blocks = max(64, N_ROWS // 5_000)   # ~5k rows per block

ds = (
    ray.data.range(N_ROWS, override_num_blocks=override_blocks)
    .map_batches(
        generate_batch,
        fn_kwargs={"seed": SEED, "categories": CATEGORIES, "embedding_dim": EMBEDDING_DIM},
        batch_size=512,
    )
    .materialize()
)
print(f"Generated {ds.count():,} rows")

# Total inline image bytes — the raw payload, used for MB/s and compression ratio.
total_image_bytes = ds.map_batches(
    lambda b: {"nbytes": np.array([sum(len(x) for x in b["image"])])},
    batch_size=512,
).sum("nbytes")
print(f"Raw image bytes: {total_image_bytes / 1e9:.3f} GB")

## Write — Parquet / Lance

Same materialized blocks → both writers. Parquet drops one file per write task with no coordination; Lance writes a fragment per task then commits fragment metadata into a new dataset version.

> **Note:** `ds.write_lance()` reports one combined write time. Splitting fragment-write vs commit time (per the README) requires the lower-level `lance.fragment.write_fragments()` + commit API — deferred here to keep the happy-path writer idiomatic.

In [ ]:
import time, os


def dir_stats(path):
    total, nfiles = 0, 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                total += os.path.getsize(os.path.join(root, f)); nfiles += 1
            except OSError:
                pass
    return total, nfiles

In [ ]:
# ── Parquet ──────────────────────────────────────────────────────────────
t0 = time.time()
ds.write_parquet(parquet_path)
parquet_write_s = time.time() - t0

pq_bytes, pq_files = dir_stats(parquet_path)
print(f"Parquet write   : {parquet_write_s:6.2f}s | "
      f"{N_ROWS / parquet_write_s:>10,.0f} rows/s | "
      f"{pq_bytes / 1e6 / parquet_write_s:6.1f} MB/s")
print(f"Parquet on-disk : {pq_bytes / 1e9:.3f} GB across {pq_files} files "
      f"({pq_bytes / max(1, pq_files) / 1e6:.1f} MB/file)")

In [ ]:
# ── Lance ────────────────────────────────────────────────────────────────
import lance

t0 = time.time()
ds.write_lance(lance_path)
lance_write_s = time.time() - t0

lds = lance.dataset(lance_path)
n_frag = len(lds.get_fragments())
lc_bytes, lc_files = dir_stats(lance_path)
print(f"Lance write     : {lance_write_s:6.2f}s | "
      f"{N_ROWS / lance_write_s:>10,.0f} rows/s | "
      f"{lc_bytes / 1e6 / lance_write_s:6.1f} MB/s")
print(f"Lance on-disk   : {lc_bytes / 1e9:.3f} GB across {n_frag} fragments")

In [ ]:
import pandas as pd

write_summary = pd.DataFrame([
    {"format": "parquet", "write_s": round(parquet_write_s, 2),
     "rows_per_s": round(N_ROWS / parquet_write_s), "on_disk_GB": round(pq_bytes / 1e9, 3),
     "files": pq_files, "compression_x": round(total_image_bytes / pq_bytes, 2)},
    {"format": "lance", "write_s": round(lance_write_s, 2),
     "rows_per_s": round(N_ROWS / lance_write_s), "on_disk_GB": round(lc_bytes / 1e9, 3),
     "files": n_frag, "compression_x": round(total_image_bytes / lc_bytes, 2)},
])
display(write_summary)

## Verify — round-trip + random access

Confirm both formats round-trip identical bytes for the same `id`, and time Lance point lookups at the start / middle / end of the dataset — access cost should be roughly constant (O(1) fragment addressing), independent of row position.

In [ ]:
import pyarrow.dataset as pads

probe_ids = [0, N_ROWS // 2, N_ROWS - 1]

# Lance point lookups + timing
print("Lance random-access latency:")
lance_rows = {}
for pid in probe_ids:
    t0 = time.time()
    row = lds.take([pid], columns=["id", "image"]).to_pylist()[0]
    lance_rows[row["id"]] = row["image"]
    print(f"  take id={pid:>12,}: {(time.time() - t0) * 1000:6.2f} ms")

# Round-trip check against Parquet
pqds = pads.dataset(parquet_path, format="parquet")
pq_tbl = pqds.to_table(filter=pads.field("id").isin(probe_ids), columns=["id", "image"]).to_pylist()
pq_map = {r["id"]: r["image"] for r in pq_tbl}

print("\nRound-trip (Parquet == Lance):")
for pid in probe_ids:
    ok = pq_map.get(pid) == lance_rows.get(pid)
    kb = len(lance_rows[pid]) / 1024
    print(f"  id={pid:>12,}: {'OK' if ok else 'MISMATCH':>8}  ({kb:.0f} KB)")

## ETL benchmark — backfill a new column

A real data-evolution operation: compute a derived column once and add it to the existing dataset. This is Lance's structural advantage — `add_columns` writes only the new column, while Parquet has no in-place column add and must rewrite the entire dataset (image bytes included).

Here the derived column is the L2 norm of the embedding — a stand-in for any UDF-computed feature.

In [ ]:
import pyarrow as pa


def compute_norm(record_batch):
    """BatchUDF: receives a pyarrow.RecordBatch, returns the new column(s)."""
    embs = np.stack(record_batch.column("embedding").to_pylist()).astype("float32")
    norms = np.linalg.norm(embs, axis=1).astype("float32")
    return pa.record_batch({"embedding_norm": pa.array(norms)})


# ── Lance: add_columns — no rewrite of existing data ───────────────────────
t0 = time.time()
lds.add_columns(compute_norm, read_columns=["embedding"])
lance_backfill_s = time.time() - t0
lc_bytes_after, _ = dir_stats(lance_path)
lance_mb_written = (lc_bytes_after - lc_bytes) / 1e6
print(f"Lance add_columns : {lance_backfill_s:6.2f}s | +{lance_mb_written:,.1f} MB (new column only)")

In [ ]:
import pyarrow.parquet as pqw

# ── Parquet: no in-place add — read all columns, append, rewrite the dataset ──
parquet_path_v2 = parquet_path + "_v2"
os.makedirs(parquet_path_v2, exist_ok=True)

t0 = time.time()
writer = None
for rb in pqds.to_batches(batch_size=2048):
    embs = np.stack(rb.column("embedding").to_pylist()).astype("float32")
    norms = np.linalg.norm(embs, axis=1).astype("float32")
    rb2 = rb.append_column("embedding_norm", pa.array(norms))
    if writer is None:
        writer = pqw.ParquetWriter(f"{parquet_path_v2}/part-0.parquet", rb2.schema)
    writer.write_batch(rb2)
if writer:
    writer.close()
parquet_backfill_s = time.time() - t0
pq_bytes_v2, _ = dir_stats(parquet_path_v2)
print(f"Parquet rewrite   : {parquet_backfill_s:6.2f}s | {pq_bytes_v2 / 1e6:,.1f} MB rewritten (whole dataset)")

In [ ]:
etl_summary = pd.DataFrame([
    {"format": "lance", "op": "add_columns", "wall_s": round(lance_backfill_s, 2),
     "MB_written": round(lance_mb_written, 1)},
    {"format": "parquet", "op": "full rewrite", "wall_s": round(parquet_backfill_s, 2),
     "MB_written": round(pq_bytes_v2 / 1e6, 1)},
])
display(etl_summary)

## Deferred write metrics

The README lists a few metrics that need infra-level instrumentation and are intentionally left out of this draft:

- **Peak worker memory** during write — needs a per-worker memory sampler.
- **Object-store PUT count** — needs cloud provider request metrics (only meaningful when writing to S3/GCS/ADLS rather than a local Volume mount).
- **Ray write-task concurrency** and **retry/error counts** — available from the Ray dashboard / logs.

---

**Next:** `02_training_benchmark.ipynb` — read each format back through Ray Data + Ray Train and measure loading + training throughput.